# BWF Match Scraper

This notebook runs the scraper and checks the generated match data.

## 1. Install the required packages

In [ ]:
%pip install -r requirements_bwf_scraper.txt

## 2. Set the year range and output folder

In [ ]:
from pathlib import Path

START_YEAR = 2021
END_YEAR = 2026
SCOPE = "world-tour"  # world-tour, elite, or all
OUTPUT_DIR = Path("bwf_data_2021_2026")
EXCLUDE_QUALIFICATION = False

print(START_YEAR, END_YEAR, SCOPE, OUTPUT_DIR)

## 3. Test with one year

Run a smaller test before collecting the full dataset.

In [ ]:
import subprocess
import sys

test_command = [
    sys.executable, "bwf_scraper_2021_2026.py",
    "--start-year", "2025",
    "--end-year", "2025",
    "--scope", SCOPE,
    "--output-dir", "bwf_test_2025",
]
subprocess.run(test_command, check=True)

## 4. Run the full scraper

In [ ]:
command = [
    sys.executable, "bwf_scraper_2021_2026.py",
    "--start-year", str(START_YEAR),
    "--end-year", str(END_YEAR),
    "--scope", SCOPE,
    "--output-dir", str(OUTPUT_DIR),
]
if EXCLUDE_QUALIFICATION:
    command.append("--exclude-qualification")

subprocess.run(command, check=True)

## 5. Load the CSV file

In [ ]:
import pandas as pd

csv_path = OUTPUT_DIR / "clean" / f"bwf_matches_{START_YEAR}_{END_YEAR}.csv"
df = pd.read_csv(csv_path, parse_dates=["date"])
print(f"Rows: {len(df):,}")
display(df.head())

## 6. Check the date range and match categories

In [ ]:
print("Date range:", df["date"].min(), "to", df["date"].max())
display(df["discipline"].value_counts().sort_index().rename("matches").to_frame())

## 7. Check doubles player IDs

In [ ]:
doubles = df[df["discipline"].isin(["MD", "WD", "XD"])].copy()

team1_duplicate = doubles[
    doubles["team1_player1_id"].notna()
    & (doubles["team1_player1_id"] == doubles["team1_player2_id"])
]
team2_duplicate = doubles[
    doubles["team2_player1_id"].notna()
    & (doubles["team2_player1_id"] == doubles["team2_player2_id"])
]

print("Doubles rows:", len(doubles))
print("Team 1 duplicate partner IDs:", len(team1_duplicate))
print("Team 2 duplicate partner IDs:", len(team2_duplicate))
assert len(team1_duplicate) == 0
assert len(team2_duplicate) == 0

## 8. View the validation summary

In [ ]:
import json

summary_path = OUTPUT_DIR / "clean" / "validation_summary.json"
with summary_path.open(encoding="utf-8") as handle:
    summary = json.load(handle)
summary

## 9. Update the dataset

Use `--refresh-index` when collecting newer tournament results.

In [ ]:
update_command = command + ["--refresh-index", "--refresh-recent-days", "45"]
print(" ".join(update_command))
# subprocess.run(update_command, check=True)  # Remove # when you want to update